# STIR-Net V1 — 22 direct instance-mask spatial micro experiments

Notebook 21 gave one positive and one negative result:

- **explicit internal cell-cell boundary supervision works**: source-9 internal-boundary AUC starts moving within a few tail-training steps;
- the previous multiclass prototype-classification loss does **not** make D1/D0 meaningfully instance-separable.

The remaining failure is more specific:

> The spatial feature space still cannot render one touching cell as a binary mask against its sibling cells.

This notebook tests that quantity **directly**.

## Core experiment

For each GT cell inside every merged current connected component:

```text
a small prototype subset of that cell
        ↓
mean feature / mask-feature vector
        ↓
dot product with held-out local voxels
        ↓
binary target:
    this GT cell = 1
    other touching GT cells = 0
```

The loss is local balanced BCE + soft Dice.

This is much closer to what STIR-Net ultimately needs than the Notebook-21 multiclass prototype-classification objective.

## Experiment arms

All arms begin from the exact same **step-30 spatial checkpoint**, and all share the same cached E2 representation.

1. `baseline`
2. `internal_boundary`
3. `raw_mask_d1`
4. `raw_mask_d0`
5. `raw_mask_d1d0`
6. `projected_mask_d1d0`
7. `internal_plus_projected_mask`

`raw_*` uses the D1/D0 feature vectors themselves.

`projected_*` first applies a learned notebook-local linear projection to `mask_dim`. This is mathematically equivalent to applying a `1×1×1` mask-feature projection at only the sampled voxels, but avoids allocating a dense 32-channel full-resolution D0 tensor.

This distinction is important:

```text
raw mask works
    → existing spatial representation can be directly shaped into instance-aware features

projected mask works but raw mask does not
    → a dedicated learned mask-feature space is likely needed

neither works
    → the E2 → D1/D0 tail itself is probably the next bottleneck
```

## Runtime

No temporal graph, CR, queries, Hungarian matching, native rendering, or full overfit.

```text
SpatialEncoder + stage_e2: run once and freeze/cache
        ↓
all 7 arms: 3 steps
        ↓
extend top 3 non-baseline arms only to step 5
```

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import copy
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.coordinates import resize_label_map_nearest
from learned.stirnet.training.checkpoint import load_checkpoint

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

SCREEN_STEPS = 3
EXTEND_TO_STEP = 5
TOP_K_EXTEND = 3

MICRO_LR = 5e-4

LAMBDA_INTERNAL_BOUNDARY = 1.0
LAMBDA_MASK_D1 = 0.75
LAMBDA_MASK_D0 = 0.75

MASK_TEMPERATURE = 0.15

# Fixed bounded supervision per GT instance.
PROTO_MAX = 64
TRAIN_POS_MAX = 128
TRAIN_NEG_MAX = 128
EVAL_POS_MAX = 256
EVAL_NEG_MAX = 256

MAX_INTERNAL_POS = 8192

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

STEP30_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "targeted"
    / "22_direct_instance_mask_micro"
)
RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook 22 requires CUDA."
    )

if not STEP30_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Exact step-30 checkpoint is missing:\n"
        f"{STEP30_CHECKPOINT}"
    )

device = torch.device("cuda")
cfg = _reduced_config()

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Checkpoint :", STEP30_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Spatial channels:", cfg.spatial.channels)
print("Mask dim        :", cfg.spatial.mask_dim)
print("Micro LR        :", MICRO_LR)

# 1. Load the same full biological scene

In [ ]:
batch, sample = build_real_batch(
    DATA_DIR
)
target = batch["targets"][0]
targets_original = batch["targets"]

spatial_inputs = batch[
    "spatial_inputs"
].to(
    device=device,
    dtype=AMP_DTYPE,
    non_blocking=True,
)

spacing_um = batch[
    "spacing_um"
].to(
    device=device,
    dtype=torch.float32,
    non_blocking=True,
)

dref_tensor = batch[
    "dref_um"
].to(
    device=device,
    dtype=torch.float32,
    non_blocking=True,
)

spatial_padding_mask = batch.get(
    "spatial_padding_mask"
)

if spatial_padding_mask is not None:
    spatial_padding_mask = (
        spatial_padding_mask.to(
            device=device,
            non_blocking=True,
        )
    )

current_labels_native = (
    batch[
        "instance_labels"
    ][0]
    .detach()
    .cpu()
    .numpy()
    .astype(
        np.int32,
        copy=False,
    )
)

gt_labels_native = (
    torch.as_tensor(
        target[
            "label_map"
        ]
    )
    .detach()
    .cpu()
    .numpy()
    .astype(
        np.int32,
        copy=False,
    )
)

spacing_native = (
    batch[
        "spacing_um"
    ][0]
    .detach()
    .cpu()
    .numpy()
    .astype(
        np.float64
    )
)

dref_um = float(
    batch[
        "dref_um"
    ][0]
)

source9_gt_ids = np.unique(
    gt_labels_native[
        current_labels_native
        == SOURCE_ID
    ]
)

source9_gt_ids = (
    source9_gt_ids[
        source9_gt_ids > 0
    ]
    .astype(
        int
    )
)

print(
    json.dumps(
        sample,
        indent=2,
        default=float,
    )
)
print(
    "Source-9 GT IDs:",
    source9_gt_ids.tolist(),
)
print(
    "Source-9 GT count:",
    len(
        source9_gt_ids
    ),
)
print(
    "dref_um:",
    dref_um,
)

# 2. Cache E2 and high-resolution encoder skips once

The notebook asks whether the **tail after E2** can learn to preserve/render distinct instances.

The encoder and `stage_e2` are therefore frozen by construction.

In [ ]:
step30_model = StirNet(
    cfg
).to(
    device
)

load_info = load_checkpoint(
    STEP30_CHECKPOINT,
    step30_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

if int(
    load_info.get(
        "step",
        -1,
    )
) != 30:
    raise RuntimeError(
        "Expected checkpoint payload step 30."
    )

step30_model.eval()

torch.cuda.reset_peak_memory_stats()
cache_start = time.perf_counter()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    ACQ_CACHE = (
        step30_model.acquisition(
            spacing_um,
            dref_tensor,
        )
        .detach()
    )

    pyramid = (
        step30_model.encoder(
            spatial_inputs,
            spacing_um,
            ACQ_CACHE,
            spatial_padding_mask,
        )
    )

    E3_CACHE = (
        pyramid.features[
            3
        ].detach()
    )

    E2_CACHE = (
        step30_model.decoder
        .decode_to_e2(
            E3_CACHE,
            pyramid,
            ACQ_CACHE,
        )
        .detach()
    )

    SKIP1_CACHE = (
        pyramid.features[
            1
        ].detach()
    )

    SKIP0_CACHE = (
        pyramid.features[
            0
        ].detach()
    )

    SPACING_D1 = (
        pyramid.spacings_um[
            1
        ]
        .detach()
        .float()
    )

    SPACING_D0 = (
        pyramid.spacings_um[
            0
        ]
        .detach()
        .float()
    )

cache_seconds = (
    time.perf_counter()
    - cache_start
)

print(
    f"Cache time: {cache_seconds:.2f}s"
)
print(
    "E2:",
    tuple(
        E2_CACHE.shape
    ),
)
print(
    "skip1:",
    tuple(
        SKIP1_CACHE.shape
    ),
)
print(
    "skip0:",
    tuple(
        SKIP0_CACHE.shape
    ),
)
print(
    "Peak CUDA:",
    round(
        torch.cuda.max_memory_allocated()
        / 1024**3,
        3,
    ),
    "GiB",
)

step30_model.cpu()

TAIL_TEMPLATE = {
    "stage_e1": copy.deepcopy(
        step30_model.decoder.stage_e1
    ),
    "stage_e0": copy.deepcopy(
        step30_model.decoder.stage_e0
    ),
    "dense_heads": copy.deepcopy(
        step30_model.dense_heads
    ),
}

del (
    step30_model,
    pyramid,
    spatial_inputs,
    E3_CACHE,
)
gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA after dropping encoder:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3,
    ),
    "GiB",
)

# 3. Downsample labels exactly to D1 and D0

In [ ]:
def labels_at_shape(
    labels_native,
    shape,
):
    return (
        resize_label_map_nearest(
            torch.from_numpy(
                labels_native.astype(
                    np.int32,
                    copy=False,
                )
            ),
            tuple(
                int(v)
                for v
                in shape
            ),
        )
        .cpu()
        .numpy()
        .astype(
            np.int32,
            copy=False,
        )
    )


D1_SHAPE = tuple(
    int(v)
    for v
    in SKIP1_CACHE.shape[
        -3:
    ]
)

D0_SHAPE = tuple(
    int(v)
    for v
    in SKIP0_CACHE.shape[
        -3:
    ]
)

GT_D1 = labels_at_shape(
    gt_labels_native,
    D1_SHAPE,
)

CURRENT_D1 = labels_at_shape(
    current_labels_native,
    D1_SHAPE,
)

GT_D0 = labels_at_shape(
    gt_labels_native,
    D0_SHAPE,
)

CURRENT_D0 = labels_at_shape(
    current_labels_native,
    D0_SHAPE,
)

print(
    "D1 shape:",
    D1_SHAPE,
)
print(
    "D0 shape:",
    D0_SHAPE,
)

# 4. Build explicit internal-boundary supervision

This is the mechanism that already showed positive movement in Notebook 21.

In [ ]:
def internal_instance_boundary(
    labels,
):
    labels = np.asarray(
        labels
    )

    boundary = np.zeros_like(
        labels,
        dtype=bool,
    )

    for axis in range(
        3
    ):
        left = [
            slice(
                None
            )
        ] * 3

        right = [
            slice(
                None
            )
        ] * 3

        left[
            axis
        ] = slice(
            0,
            -1,
        )

        right[
            axis
        ] = slice(
            1,
            None,
        )

        a = labels[
            tuple(
                left
            )
        ]

        b = labels[
            tuple(
                right
            )
        ]

        different = (
            (a > 0)
            & (b > 0)
            & (a != b)
        )

        boundary[
            tuple(
                left
            )
        ] |= different

        boundary[
            tuple(
                right
            )
        ] |= different

    return boundary


def physical_dilate(
    mask,
    spacing,
    width_um=1.0,
):
    spacing = np.asarray(
        spacing,
        dtype=np.float64,
    )

    radius = np.ceil(
        float(
            width_um
        )
        / spacing
    ).astype(
        int
    )

    zz, yy, xx = np.ogrid[
        -radius[0]:
        radius[0] + 1,
        -radius[1]:
        radius[1] + 1,
        -radius[2]:
        radius[2] + 1,
    ]

    structure = (
        (
            zz
            * spacing[
                0
            ]
        )
        ** 2
        + (
            yy
            * spacing[
                1
            ]
        )
        ** 2
        + (
            xx
            * spacing[
                2
            ]
        )
        ** 2
        <= float(
            width_um
        )
        ** 2
    )

    return ndi.binary_dilation(
        mask,
        structure=structure,
    )


INTERNAL_RAW_D0 = (
    internal_instance_boundary(
        GT_D0
    )
)

INTERNAL_D0 = (
    physical_dilate(
        INTERNAL_RAW_D0,
        SPACING_D0[
            0
        ].cpu().numpy(),
        width_um=1.0,
    )
)

FOREGROUND_D0 = (
    GT_D0 > 0
)


def deterministic_choice(
    indices,
    count,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(
        indices
    ) <= int(
        count
    ):
        return indices

    rng = (
        np.random.default_rng(
            int(
                seed
            )
        )
    )

    selected = rng.choice(
        indices,
        size=int(
            count
        ),
        replace=False,
    )

    return np.sort(
        selected
    )


@dataclass
class BinarySample:
    indices: torch.Tensor
    targets: torch.Tensor


positive_indices = np.flatnonzero(
    INTERNAL_D0.reshape(
        -1
    )
)

negative_indices = np.flatnonzero(
    (
        FOREGROUND_D0
        & ~INTERNAL_D0
    ).reshape(
        -1
    )
)

positive_indices = (
    deterministic_choice(
        positive_indices,
        min(
            len(
                positive_indices
            ),
            MAX_INTERNAL_POS,
        ),
        SEED + 101,
    )
)

negative_indices = (
    deterministic_choice(
        negative_indices,
        len(
            positive_indices
        ),
        SEED + 102,
    )
)

internal_indices = np.concatenate(
    [
        positive_indices,
        negative_indices,
    ]
)

internal_targets = np.concatenate(
    [
        np.ones(
            len(
                positive_indices
            ),
            dtype=np.float32,
        ),
        np.zeros(
            len(
                negative_indices
            ),
            dtype=np.float32,
        ),
    ]
)

rng = np.random.default_rng(
    SEED + 103
)

order = rng.permutation(
    len(
        internal_indices
    )
)

INTERNAL_SAMPLE_D0 = BinarySample(
    indices=torch.from_numpy(
        internal_indices[
            order
        ]
    ).long(),
    targets=torch.from_numpy(
        internal_targets[
            order
        ]
    ).float(),
)

print(
    "Internal D0 sample:",
    int(
        (
            INTERNAL_SAMPLE_D0.targets
            == 1
        ).sum()
    ),
    "positive +",
    int(
        (
            INTERNAL_SAMPLE_D0.targets
            == 0
        ).sum()
    ),
    "negative",
)

# 5. Build true binary instance-mask tasks

Each task represents **one GT cell inside one merged current component**.

The positive GT-cell voxels are split into three disjoint subsets:

```text
prototype subset
training-positive subset
evaluation-positive subset
```

Hard negatives are voxels belonging to the *other GT cells inside the same current merged component*.

This is exactly the distinction the model currently fails to make.

In [ ]:
@dataclass
class InstanceMaskTask:
    source_id: int
    gt_id: int
    proto_indices: torch.Tensor
    train_pos_indices: torch.Tensor
    train_neg_indices: torch.Tensor
    eval_pos_indices: torch.Tensor
    eval_neg_indices: torch.Tensor


def split_fixed(
    indices,
    *,
    proto_max,
    train_max,
    eval_max,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    rng = np.random.default_rng(
        int(
            seed
        )
    )

    indices = indices[
        rng.permutation(
            len(
                indices
            )
        )
    ]

    n_proto = min(
        int(
            proto_max
        ),
        max(
            2,
            len(
                indices
            )
            // 4,
        ),
    )

    remaining = (
        len(
            indices
        )
        - n_proto
    )

    n_train = min(
        int(
            train_max
        ),
        max(
            2,
            remaining
            // 2,
        ),
    )

    remaining = (
        remaining
        - n_train
    )

    n_eval = min(
        int(
            eval_max
        ),
        remaining,
    )

    if (
        n_proto < 2
        or n_train < 2
        or n_eval < 2
    ):
        return None

    return (
        indices[
            :n_proto
        ],
        indices[
            n_proto:
            n_proto + n_train
        ],
        indices[
            n_proto + n_train:
            n_proto + n_train + n_eval
        ],
    )


def build_instance_mask_tasks(
    current_labels,
    gt_labels,
    *,
    seed,
):
    current_flat = (
        current_labels.reshape(
            -1
        )
    )

    gt_flat = (
        gt_labels.reshape(
            -1
        )
    )

    tasks = []

    for source_id in np.unique(
        current_flat
    ):
        if source_id <= 0:
            continue

        source_positions = np.flatnonzero(
            current_flat
            == int(
                source_id
            )
        )

        overlapping_gt = np.unique(
            gt_flat[
                source_positions
            ]
        )

        overlapping_gt = (
            overlapping_gt[
                overlapping_gt > 0
            ]
        )

        if len(
            overlapping_gt
        ) < 2:
            continue

        for gt_id in overlapping_gt:
            positive = np.flatnonzero(
                (
                    current_flat
                    == int(
                        source_id
                    )
                )
                & (
                    gt_flat
                    == int(
                        gt_id
                    )
                )
            )

            hard_negative = np.flatnonzero(
                (
                    current_flat
                    == int(
                        source_id
                    )
                )
                & (
                    gt_flat > 0
                )
                & (
                    gt_flat
                    != int(
                        gt_id
                    )
                )
            )

            positive_split = split_fixed(
                positive,
                proto_max=PROTO_MAX,
                train_max=TRAIN_POS_MAX,
                eval_max=EVAL_POS_MAX,
                seed=(
                    seed
                    + 1009
                    * int(
                        source_id
                    )
                    + 37
                    * int(
                        gt_id
                    )
                ),
            )

            if positive_split is None:
                continue

            (
                proto_idx,
                train_pos,
                eval_pos,
            ) = positive_split

            negative_need = (
                TRAIN_NEG_MAX
                + EVAL_NEG_MAX
            )

            if len(
                hard_negative
            ) < 4:
                continue

            rng = np.random.default_rng(
                int(
                    seed
                    + 2003
                    * int(
                        source_id
                    )
                    + 53
                    * int(
                        gt_id
                    )
                )
            )

            hard_negative = (
                hard_negative[
                    rng.permutation(
                        len(
                            hard_negative
                        )
                    )
                ]
            )

            n_train_neg = min(
                int(
                    TRAIN_NEG_MAX
                ),
                max(
                    2,
                    len(
                        hard_negative
                    )
                    // 2,
                ),
            )

            remaining_neg = (
                len(
                    hard_negative
                )
                - n_train_neg
            )

            n_eval_neg = min(
                int(
                    EVAL_NEG_MAX
                ),
                remaining_neg,
            )

            if (
                n_train_neg < 2
                or n_eval_neg < 2
            ):
                continue

            train_neg = (
                hard_negative[
                    :n_train_neg
                ]
            )

            eval_neg = (
                hard_negative[
                    n_train_neg:
                    n_train_neg
                    + n_eval_neg
                ]
            )

            tasks.append(
                InstanceMaskTask(
                    source_id=int(
                        source_id
                    ),
                    gt_id=int(
                        gt_id
                    ),
                    proto_indices=torch.from_numpy(
                        proto_idx
                    ).long(),
                    train_pos_indices=torch.from_numpy(
                        train_pos
                    ).long(),
                    train_neg_indices=torch.from_numpy(
                        train_neg
                    ).long(),
                    eval_pos_indices=torch.from_numpy(
                        eval_pos
                    ).long(),
                    eval_neg_indices=torch.from_numpy(
                        eval_neg
                    ).long(),
                )
            )

    return tasks


TASKS_D1 = build_instance_mask_tasks(
    CURRENT_D1,
    GT_D1,
    seed=SEED + 1000,
)

TASKS_D0 = build_instance_mask_tasks(
    CURRENT_D0,
    GT_D0,
    seed=SEED + 2000,
)

print(
    "D1 tasks:",
    len(
        TASKS_D1
    ),
)
print(
    "D0 tasks:",
    len(
        TASKS_D0
    ),
)

print(
    "Source-9 D1 tasks:",
    sum(
        task.source_id
        == SOURCE_ID
        for task in TASKS_D1
    ),
)
print(
    "Source-9 D0 tasks:",
    sum(
        task.source_id
        == SOURCE_ID
        for task in TASKS_D0
    ),
)

if not any(
    task.source_id
    == SOURCE_ID
    for task in TASKS_D1
):
    raise RuntimeError(
        "No source-9 D1 mask tasks were built."
    )

if not any(
    task.source_id
    == SOURCE_ID
    for task in TASKS_D0
):
    raise RuntimeError(
        "No source-9 D0 mask tasks were built."
    )

# 6. Tail branch

The notebook-local `Linear(C, mask_dim, bias=False)` projectors are applied only to sampled voxel features.

They are equivalent to a `1×1×1` convolution evaluated at those voxels and are intentionally cheap.

In [ ]:
class TailBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.stage_e1 = copy.deepcopy(
            TAIL_TEMPLATE[
                "stage_e1"
            ]
        )

        self.stage_e0 = copy.deepcopy(
            TAIL_TEMPLATE[
                "stage_e0"
            ]
        )

        self.dense_heads = copy.deepcopy(
            TAIL_TEMPLATE[
                "dense_heads"
            ]
        )

        d1_channels = int(
            cfg.spatial.channels[
                1
            ]
        )

        d0_channels = int(
            cfg.spatial.channels[
                0
            ]
        )

        mask_dim = int(
            cfg.spatial.mask_dim
        )

        self.mask_proj_d1 = nn.Linear(
            d1_channels,
            mask_dim,
            bias=False,
        )

        self.mask_proj_d0 = nn.Linear(
            d0_channels,
            mask_dim,
            bias=False,
        )

    def forward(self):
        d1 = self.stage_e1(
            E2_CACHE,
            SKIP1_CACHE,
            ACQ_CACHE,
        )

        d0 = self.stage_e0(
            d1,
            SKIP0_CACHE,
            ACQ_CACHE,
        )

        dense = self.dense_heads(
            d0
        )

        return {
            "d1": d1,
            "d0": d0,
            "dense": dense,
        }


criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(
    device
)


def build_fresh_branch():
    torch.manual_seed(
        SEED + 9001
    )
    torch.cuda.manual_seed_all(
        SEED + 9001
    )

    return TailBranch().to(
        device
    )

# 7. Loss helpers

In [ ]:
def gather_feature_rows(
    feature,
    indices_cpu,
):
    flat = (
        feature[
            0
        ]
        .flatten(
            1
        )
        .transpose(
            0,
            1,
        )
    )

    index = indices_cpu.to(
        device=flat.device,
        non_blocking=True,
    )

    return flat[
        index
    ]


def balanced_binary_mask_loss(
    logits,
    targets,
):
    logits = logits.float()

    targets = targets.to(
        device=logits.device,
        dtype=torch.float32,
        non_blocking=True,
    )

    bce = (
        F.binary_cross_entropy_with_logits(
            logits,
            targets,
        )
    )

    probability = (
        logits.sigmoid()
    )

    intersection = (
        probability
        * targets
    ).sum()

    soft_dice_loss = (
        1.0
        - (
            2.0
            * intersection
            + 1e-6
        )
        / (
            probability.sum()
            + targets.sum()
            + 1e-6
        )
    )

    return (
        bce
        + soft_dice_loss
    )


def transform_feature(
    rows,
    projector,
):
    if projector is None:
        return rows.float()

    return projector(
        rows.float()
    )


def task_logits(
    feature,
    task,
    *,
    projector,
    split,
):
    proto_rows = gather_feature_rows(
        feature,
        task.proto_indices,
    )

    if split == "train":
        positive_indices = (
            task.train_pos_indices
        )
        negative_indices = (
            task.train_neg_indices
        )
    elif split == "eval":
        positive_indices = (
            task.eval_pos_indices
        )
        negative_indices = (
            task.eval_neg_indices
        )
    else:
        raise ValueError(
            split
        )

    positive_rows = gather_feature_rows(
        feature,
        positive_indices,
    )

    negative_rows = gather_feature_rows(
        feature,
        negative_indices,
    )

    proto_rows = transform_feature(
        proto_rows,
        projector,
    )

    positive_rows = transform_feature(
        positive_rows,
        projector,
    )

    negative_rows = transform_feature(
        negative_rows,
        projector,
    )

    prototype = (
        F.normalize(
            proto_rows,
            dim=-1,
        )
        .mean(
            dim=0
        )
    )

    prototype = F.normalize(
        prototype,
        dim=-1,
    )

    candidates = torch.cat(
        [
            positive_rows,
            negative_rows,
        ],
        dim=0,
    )

    candidates = F.normalize(
        candidates,
        dim=-1,
    )

    logits = (
        candidates
        @ prototype
        / float(
            MASK_TEMPERATURE
        )
    )

    targets = torch.cat(
        [
            torch.ones(
                len(
                    positive_rows
                ),
                device=feature.device,
            ),
            torch.zeros(
                len(
                    negative_rows
                ),
                device=feature.device,
            ),
        ]
    )

    return (
        logits,
        targets,
    )


def direct_instance_mask_loss(
    feature,
    tasks,
    *,
    projector,
):
    losses = []

    for task in tasks:
        logits, targets = (
            task_logits(
                feature,
                task,
                projector=projector,
                split="train",
            )
        )

        losses.append(
            balanced_binary_mask_loss(
                logits,
                targets,
            )
        )

    if not losses:
        return (
            feature.sum()
            * 0.0
        )

    return torch.stack(
        losses
    ).mean()


def sampled_boundary_logits(
    boundary_logits,
):
    flat = (
        boundary_logits[
            0,
            0
        ].reshape(
            -1
        )
    )

    index = (
        INTERNAL_SAMPLE_D0
        .indices
        .to(
            device=flat.device,
            non_blocking=True,
        )
    )

    return flat[
        index
    ]


def internal_boundary_loss(
    dense,
):
    logits = sampled_boundary_logits(
        dense[
            "boundary_logits"
        ]
    )

    return balanced_binary_mask_loss(
        logits,
        INTERNAL_SAMPLE_D0.targets,
    )


def original_dense_loss(
    dense,
):
    (
        foreground,
        center,
        boundary,
    ) = criterion._dense_losses(
        dense,
        targets_original,
    )

    total = (
        float(
            cfg.losses.foreground
        )
        * foreground
        + float(
            cfg.losses.center_heatmap
        )
        * center
        + float(
            cfg.losses.boundary
        )
        * boundary
    )

    return {
        "base_total": total,
        "foreground": foreground,
        "center": center,
        "boundary": boundary,
    }

# 8. Define independent experiment arms

In [ ]:
ARM_SPECS = {
    "baseline": {
        "internal": False,
        "raw_d1": False,
        "raw_d0": False,
        "projected_d1": False,
        "projected_d0": False,
    },
    "internal_boundary": {
        "internal": True,
        "raw_d1": False,
        "raw_d0": False,
        "projected_d1": False,
        "projected_d0": False,
    },
    "raw_mask_d1": {
        "internal": False,
        "raw_d1": True,
        "raw_d0": False,
        "projected_d1": False,
        "projected_d0": False,
    },
    "raw_mask_d0": {
        "internal": False,
        "raw_d1": False,
        "raw_d0": True,
        "projected_d1": False,
        "projected_d0": False,
    },
    "raw_mask_d1d0": {
        "internal": False,
        "raw_d1": True,
        "raw_d0": True,
        "projected_d1": False,
        "projected_d0": False,
    },
    "projected_mask_d1d0": {
        "internal": False,
        "raw_d1": False,
        "raw_d0": False,
        "projected_d1": True,
        "projected_d0": True,
    },
    "internal_plus_projected_mask": {
        "internal": True,
        "raw_d1": False,
        "raw_d0": False,
        "projected_d1": True,
        "projected_d0": True,
    },
}

display(
    pd.DataFrame(
        ARM_SPECS
    ).T
)

# 9. Arm objective

In [ ]:
def arm_objective(
    branch,
    outputs,
    spec,
):
    base = original_dense_loss(
        outputs[
            "dense"
        ]
    )

    zero = (
        base[
            "base_total"
        ]
        * 0.0
    )

    loss_internal = zero
    raw_d1 = zero
    raw_d0 = zero
    projected_d1 = zero
    projected_d0 = zero

    if spec[
        "internal"
    ]:
        loss_internal = (
            internal_boundary_loss(
                outputs[
                    "dense"
                ]
            )
        )

    if spec[
        "raw_d1"
    ]:
        raw_d1 = (
            direct_instance_mask_loss(
                outputs[
                    "d1"
                ],
                TASKS_D1,
                projector=None,
            )
        )

    if spec[
        "raw_d0"
    ]:
        raw_d0 = (
            direct_instance_mask_loss(
                outputs[
                    "d0"
                ],
                TASKS_D0,
                projector=None,
            )
        )

    if spec[
        "projected_d1"
    ]:
        projected_d1 = (
            direct_instance_mask_loss(
                outputs[
                    "d1"
                ],
                TASKS_D1,
                projector=(
                    branch.mask_proj_d1
                ),
            )
        )

    if spec[
        "projected_d0"
    ]:
        projected_d0 = (
            direct_instance_mask_loss(
                outputs[
                    "d0"
                ],
                TASKS_D0,
                projector=(
                    branch.mask_proj_d0
                ),
            )
        )

    total = (
        base[
            "base_total"
        ]
        + float(
            LAMBDA_INTERNAL_BOUNDARY
        )
        * loss_internal
        + float(
            LAMBDA_MASK_D1
        )
        * (
            raw_d1
            + projected_d1
        )
        + float(
            LAMBDA_MASK_D0
        )
        * (
            raw_d0
            + projected_d0
        )
    )

    return {
        "loss": total,
        "base_total": (
            base[
                "base_total"
            ]
        ),
        "foreground": (
            base[
                "foreground"
            ]
        ),
        "center": (
            base[
                "center"
            ]
        ),
        "boundary": (
            base[
                "boundary"
            ]
        ),
        "internal": (
            loss_internal
        ),
        "raw_mask_d1": (
            raw_d1
        ),
        "raw_mask_d0": (
            raw_d0
        ),
        "projected_mask_d1": (
            projected_d1
        ),
        "projected_mask_d0": (
            projected_d0
        ),
    }

# 10. Evaluation metrics

The direct mask metric is computed on the **disjoint evaluation subsets** that were never used by the direct mask loss.

For source 9, each of the nine GT cells gets its own binary mask score against hard negatives from its touching siblings.

In [ ]:
def binary_auc(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    positive = (
        labels == 1
    )

    negative = (
        labels == 0
    )

    if (
        positive.sum() == 0
        or negative.sum() == 0
    ):
        return float(
            "nan"
        )

    ranks = rankdata(
        scores
    )

    n_pos = int(
        positive.sum()
    )

    n_neg = int(
        negative.sum()
    )

    return float(
        (
            ranks[
                positive
            ].sum()
            - n_pos
            * (
                n_pos
                + 1
            )
            / 2
        )
        / (
            n_pos
            * n_neg
        )
    )


def source9_boundary_metrics(
    boundary_logits,
):
    probability = (
        boundary_logits[
            0,
            0
        ]
        .detach()
        .float()
        .sigmoid()
        .cpu()
        .numpy()
    )

    source = (
        CURRENT_D0
        == SOURCE_ID
    )

    positive = (
        INTERNAL_RAW_D0
        & source
    )

    negative = (
        source
        & ~positive
    )

    positive_scores = (
        probability[
            positive
        ]
    )

    negative_scores = (
        probability[
            negative
        ]
    )

    scores = np.concatenate(
        [
            positive_scores,
            negative_scores,
        ]
    )

    labels = np.concatenate(
        [
            np.ones(
                len(
                    positive_scores
                ),
                dtype=np.int64,
            ),
            np.zeros(
                len(
                    negative_scores
                ),
                dtype=np.int64,
            ),
        ]
    )

    return {
        "source9_internal_auc": (
            binary_auc(
                scores,
                labels,
            )
        ),
        "source9_internal_recall_0p5": float(
            (
                positive_scores
                >= 0.5
            ).mean()
        ),
        "source9_internal_prob": float(
            positive_scores.mean()
        ),
        "source9_nonboundary_prob": float(
            negative_scores.mean()
        ),
    }


def eval_direct_mask_tasks(
    feature,
    tasks,
    *,
    projector,
    source_id=None,
):
    dice_values = []
    auc_values = []

    for task in tasks:
        if (
            source_id is not None
            and int(
                task.source_id
            )
            != int(
                source_id
            )
        ):
            continue

        logits, targets = (
            task_logits(
                feature,
                task,
                projector=projector,
                split="eval",
            )
        )

        probability = (
            logits.sigmoid()
        )

        intersection = (
            probability
            * targets
        ).sum()

        dice = (
            2.0
            * intersection
            + 1e-6
        ) / (
            probability.sum()
            + targets.sum()
            + 1e-6
        )

        dice_values.append(
            float(
                dice.detach()
                .cpu()
            )
        )

        auc_values.append(
            binary_auc(
                logits.detach()
                .float()
                .cpu()
                .numpy(),
                targets.detach()
                .cpu()
                .numpy()
                .astype(
                    np.int64
                ),
            )
        )

    if not dice_values:
        return {
            "dice": float(
                "nan"
            ),
            "auc": float(
                "nan"
            ),
            "task_count": 0,
        }

    return {
        "dice": float(
            np.mean(
                dice_values
            )
        ),
        "auc": float(
            np.nanmean(
                auc_values
            )
        ),
        "task_count": int(
            len(
                dice_values
            )
        ),
    }


@torch.no_grad()
def evaluate_branch(
    branch,
    *,
    arm,
    step,
):
    branch.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = branch()

        base = original_dense_loss(
            outputs[
                "dense"
            ]
        )

    boundary = (
        source9_boundary_metrics(
            outputs[
                "dense"
            ][
                "boundary_logits"
            ]
        )
    )

    raw_d1 = eval_direct_mask_tasks(
        outputs[
            "d1"
        ],
        TASKS_D1,
        projector=None,
        source_id=SOURCE_ID,
    )

    raw_d0 = eval_direct_mask_tasks(
        outputs[
            "d0"
        ],
        TASKS_D0,
        projector=None,
        source_id=SOURCE_ID,
    )

    projected_d1 = (
        eval_direct_mask_tasks(
            outputs[
                "d1"
            ],
            TASKS_D1,
            projector=(
                branch.mask_proj_d1
            ),
            source_id=SOURCE_ID,
        )
    )

    projected_d0 = (
        eval_direct_mask_tasks(
            outputs[
                "d0"
            ],
            TASKS_D0,
            projector=(
                branch.mask_proj_d0
            ),
            source_id=SOURCE_ID,
        )
    )

    result = {
        "arm": arm,
        "step": int(
            step
        ),
        "original_base_loss": float(
            base[
                "base_total"
            ]
            .detach()
            .cpu()
        ),
        "foreground_loss": float(
            base[
                "foreground"
            ]
            .detach()
            .cpu()
        ),
        "center_loss": float(
            base[
                "center"
            ]
            .detach()
            .cpu()
        ),
        "boundary_loss": float(
            base[
                "boundary"
            ]
            .detach()
            .cpu()
        ),
        **boundary,
        "raw_d1_mask_dice": (
            raw_d1[
                "dice"
            ]
        ),
        "raw_d1_mask_auc": (
            raw_d1[
                "auc"
            ]
        ),
        "raw_d0_mask_dice": (
            raw_d0[
                "dice"
            ]
        ),
        "raw_d0_mask_auc": (
            raw_d0[
                "auc"
            ]
        ),
        "projected_d1_mask_dice": (
            projected_d1[
                "dice"
            ]
        ),
        "projected_d1_mask_auc": (
            projected_d1[
                "auc"
            ]
        ),
        "projected_d0_mask_dice": (
            projected_d0[
                "dice"
            ]
        ),
        "projected_d0_mask_auc": (
            projected_d0[
                "auc"
            ]
        ),
    }

    del outputs

    return result

# 11. Common step-0 baseline

In [ ]:
zero_branch = (
    build_fresh_branch()
)

zero_metrics = (
    evaluate_branch(
        zero_branch,
        arm="common_step0",
        step=0,
    )
)

display(
    pd.DataFrame(
        [
            zero_metrics
        ]
    )
)

del zero_branch
gc.collect()
torch.cuda.empty_cache()

# 12. Micro-training runner

We also record gradient norms by subsystem. This matters if a loss moves its own scalar but fails to send useful gradients into `stage_e1` / `stage_e0`.

In [ ]:
def module_grad_norm(
    module,
):
    total = 0.0
    finite = True

    for parameter in module.parameters():
        if parameter.grad is None:
            continue

        grad = parameter.grad.detach()

        finite = (
            finite
            and bool(
                torch.isfinite(
                    grad
                ).all()
            )
        )

        total += float(
            grad.float()
            .square()
            .sum()
            .detach()
            .cpu()
        )

    return (
        math.sqrt(
            max(
                total,
                0.0,
            )
        ),
        finite,
    )


def cpu_state_dict(
    module,
):
    return {
        key: value.detach()
        .cpu()
        .clone()
        for key, value
        in module.state_dict().items()
    }


def run_arm(
    arm,
    spec,
    *,
    total_steps,
    initial_state=None,
    starting_step=0,
):
    branch = (
        build_fresh_branch()
    )

    if initial_state is not None:
        branch.load_state_dict(
            initial_state,
            strict=True,
        )

    optimizer = (
        torch.optim.AdamW(
            branch.parameters(),
            lr=float(
                MICRO_LR
            ),
            weight_decay=float(
                cfg.training.weight_decay
            ),
        )
    )

    scaler = (
        torch.amp.GradScaler(
            "cuda",
            enabled=True,
            init_scale=1024.0,
        )
    )

    metric_rows = []
    train_rows = []

    eval_points = {
        int(
            starting_step
        ),
        int(
            starting_step
            + 1
        ),
        int(
            total_steps
        ),
    }

    if (
        starting_step == 0
        and total_steps >= 3
    ):
        eval_points.add(
            3
        )

    branch_start = (
        time.perf_counter()
    )

    for step in range(
        int(
            starting_step
        ),
        int(
            total_steps
        )
        + 1,
    ):
        if step in eval_points:
            metric_rows.append(
                evaluate_branch(
                    branch,
                    arm=arm,
                    step=step,
                )
            )

        if step == total_steps:
            break

        branch.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        torch.cuda.reset_peak_memory_stats()

        step_start = (
            time.perf_counter()
        )

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            outputs = branch()

            objective = (
                arm_objective(
                    branch,
                    outputs,
                    spec,
                )
            )

        scaler.scale(
            objective[
                "loss"
            ]
        ).backward()

        scaler.unscale_(
            optimizer
        )

        e1_grad, e1_finite = (
            module_grad_norm(
                branch.stage_e1
            )
        )

        e0_grad, e0_finite = (
            module_grad_norm(
                branch.stage_e0
            )
        )

        dense_grad, dense_finite = (
            module_grad_norm(
                branch.dense_heads
            )
        )

        p1_grad, p1_finite = (
            module_grad_norm(
                branch.mask_proj_d1
            )
        )

        p0_grad, p0_finite = (
            module_grad_norm(
                branch.mask_proj_d0
            )
        )

        finite = bool(
            e1_finite
            and e0_finite
            and dense_finite
            and p1_finite
            and p0_finite
        )

        torch.nn.utils.clip_grad_norm_(
            branch.parameters(),
            float(
                cfg.training.max_grad_norm
            ),
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        train_rows.append({
            "arm": arm,
            "from_step": int(
                step
            ),
            "to_step": int(
                step + 1
            ),
            "loss": float(
                objective[
                    "loss"
                ]
                .detach()
                .cpu()
            ),
            "base_total": float(
                objective[
                    "base_total"
                ]
                .detach()
                .cpu()
            ),
            "foreground": float(
                objective[
                    "foreground"
                ]
                .detach()
                .cpu()
            ),
            "center": float(
                objective[
                    "center"
                ]
                .detach()
                .cpu()
            ),
            "boundary": float(
                objective[
                    "boundary"
                ]
                .detach()
                .cpu()
            ),
            "internal": float(
                objective[
                    "internal"
                ]
                .detach()
                .cpu()
            ),
            "raw_mask_d1": float(
                objective[
                    "raw_mask_d1"
                ]
                .detach()
                .cpu()
            ),
            "raw_mask_d0": float(
                objective[
                    "raw_mask_d0"
                ]
                .detach()
                .cpu()
            ),
            "projected_mask_d1": float(
                objective[
                    "projected_mask_d1"
                ]
                .detach()
                .cpu()
            ),
            "projected_mask_d0": float(
                objective[
                    "projected_mask_d0"
                ]
                .detach()
                .cpu()
            ),
            "stage_e1_grad_norm": float(
                e1_grad
            ),
            "stage_e0_grad_norm": float(
                e0_grad
            ),
            "dense_grad_norm": float(
                dense_grad
            ),
            "proj_d1_grad_norm": float(
                p1_grad
            ),
            "proj_d0_grad_norm": float(
                p0_grad
            ),
            "finite_gradients": finite,
            "step_seconds": float(
                time.perf_counter()
                - step_start
            ),
            "peak_cuda_gib": float(
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
        })

        del (
            outputs,
            objective,
        )

    final_state = (
        cpu_state_dict(
            branch
        )
    )

    elapsed = (
        time.perf_counter()
        - branch_start
    )

    del (
        branch,
        optimizer,
        scaler,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "metrics": pd.DataFrame(
            metric_rows
        ),
        "training": pd.DataFrame(
            train_rows
        ),
        "state": final_state,
        "elapsed_s": float(
            elapsed
        ),
    }

# 13. Three-step independent screen of all seven mechanisms

In [ ]:
SCREEN_RUNS = {}

metric_frames = []
training_frames = []

for arm, spec in ARM_SPECS.items():
    print(
        "\n"
        + "=" * 80
    )
    print(
        "SCREEN:",
        arm
    )
    print(
        "=" * 80
    )

    result = run_arm(
        arm,
        spec,
        total_steps=SCREEN_STEPS,
        initial_state=None,
        starting_step=0,
    )

    SCREEN_RUNS[
        arm
    ] = result

    metric_frames.append(
        result[
            "metrics"
        ]
    )

    training_frames.append(
        result[
            "training"
        ]
    )

    display(
        result[
            "metrics"
        ][
            [
                "arm",
                "step",
                "source9_internal_auc",
                "raw_d1_mask_dice",
                "raw_d0_mask_dice",
                "projected_d1_mask_dice",
                "projected_d0_mask_dice",
                "foreground_loss",
            ]
        ]
    )

    print(
        "elapsed:",
        round(
            result[
                "elapsed_s"
            ],
            2,
        ),
        "s",
    )


screen_metrics_df = pd.concat(
    metric_frames,
    ignore_index=True,
)

screen_training_df = pd.concat(
    training_frames,
    ignore_index=True,
)

screen_metrics_df.to_csv(
    RUN_DIR
    / "screen_metrics.csv",
    index=False,
)

screen_training_df.to_csv(
    RUN_DIR
    / "screen_training.csv",
    index=False,
)

# 14. Rank mechanisms at step 3

The score is only used to decide which three arms deserve two additional steps.

Raw measurements remain the evidence.

In [ ]:
step3 = (
    screen_metrics_df[
        screen_metrics_df[
            "step"
        ]
        == SCREEN_STEPS
    ]
    .copy()
)

base_raw = 0.5 * (
    float(
        zero_metrics[
            "raw_d1_mask_dice"
        ]
    )
    + float(
        zero_metrics[
            "raw_d0_mask_dice"
        ]
    )
)

base_projected = 0.5 * (
    float(
        zero_metrics[
            "projected_d1_mask_dice"
        ]
    )
    + float(
        zero_metrics[
            "projected_d0_mask_dice"
        ]
    )
)

base_boundary_auc = float(
    zero_metrics[
        "source9_internal_auc"
    ]
)

base_foreground = float(
    zero_metrics[
        "foreground_loss"
    ]
)

step3[
    "raw_mask_mean"
] = 0.5 * (
    step3[
        "raw_d1_mask_dice"
    ]
    + step3[
        "raw_d0_mask_dice"
    ]
)

step3[
    "projected_mask_mean"
] = 0.5 * (
    step3[
        "projected_d1_mask_dice"
    ]
    + step3[
        "projected_d0_mask_dice"
    ]
)

step3[
    "raw_mask_gain"
] = (
    step3[
        "raw_mask_mean"
    ]
    - base_raw
)

step3[
    "projected_mask_gain"
] = (
    step3[
        "projected_mask_mean"
    ]
    - base_projected
)

step3[
    "best_instance_gain"
] = np.maximum(
    step3[
        "raw_mask_gain"
    ],
    step3[
        "projected_mask_gain"
    ],
)

step3[
    "boundary_auc_gain"
] = (
    step3[
        "source9_internal_auc"
    ]
    - base_boundary_auc
)

step3[
    "foreground_ratio"
] = (
    step3[
        "foreground_loss"
    ]
    / max(
        base_foreground,
        1e-8,
    )
)

step3[
    "diagnostic_score"
] = (
    3.0
    * step3[
        "best_instance_gain"
    ]
    + 1.0
    * step3[
        "boundary_auc_gain"
    ]
    - 0.25
    * np.maximum(
        0.0,
        step3[
            "foreground_ratio"
        ]
        - 1.05,
    )
)

step3 = step3.sort_values(
    "diagnostic_score",
    ascending=False,
)

display(
    step3[
        [
            "arm",
            "source9_internal_auc",
            "boundary_auc_gain",
            "raw_d1_mask_dice",
            "raw_d0_mask_dice",
            "raw_mask_gain",
            "projected_d1_mask_dice",
            "projected_d0_mask_dice",
            "projected_mask_gain",
            "best_instance_gain",
            "foreground_ratio",
            "diagnostic_score",
        ]
    ]
)

step3.to_csv(
    RUN_DIR
    / "step3_comparison.csv",
    index=False,
)

# 15. Extend only the top three non-baseline arms to step 5

In [ ]:
TOP_ARMS = (
    step3[
        step3[
            "arm"
        ]
        != "baseline"
    ][
        "arm"
    ]
    .head(
        TOP_K_EXTEND
    )
    .tolist()
)

print(
    "Extending:",
    TOP_ARMS,
)

EXTENSION_RUNS = {}
extension_metric_frames = []
extension_training_frames = []

for arm in TOP_ARMS:
    print(
        "\n"
        + "=" * 80
    )
    print(
        "EXTEND:",
        arm,
        f"{SCREEN_STEPS} → {EXTEND_TO_STEP}"
    )
    print(
        "=" * 80
    )

    result = run_arm(
        arm,
        ARM_SPECS[
            arm
        ],
        total_steps=EXTEND_TO_STEP,
        initial_state=(
            SCREEN_RUNS[
                arm
            ][
                "state"
            ]
        ),
        starting_step=SCREEN_STEPS,
    )

    EXTENSION_RUNS[
        arm
    ] = result

    extension_metric_frames.append(
        result[
            "metrics"
        ]
    )

    extension_training_frames.append(
        result[
            "training"
        ]
    )

    display(
        result[
            "metrics"
        ][
            [
                "arm",
                "step",
                "source9_internal_auc",
                "raw_d1_mask_dice",
                "raw_d0_mask_dice",
                "projected_d1_mask_dice",
                "projected_d0_mask_dice",
                "foreground_loss",
            ]
        ]
    )


extension_metrics_df = (
    pd.concat(
        extension_metric_frames,
        ignore_index=True,
    )
    if extension_metric_frames
    else pd.DataFrame()
)

extension_training_df = (
    pd.concat(
        extension_training_frames,
        ignore_index=True,
    )
    if extension_training_frames
    else pd.DataFrame()
)

extension_metrics_df.to_csv(
    RUN_DIR
    / "extension_metrics.csv",
    index=False,
)

extension_training_df.to_csv(
    RUN_DIR
    / "extension_training.csv",
    index=False,
)

# 16. Final arm table

In [ ]:
final_rows = []

for arm in ARM_SPECS:
    selected = None

    if arm in TOP_ARMS:
        rows = extension_metrics_df[
            (
                extension_metrics_df[
                    "arm"
                ]
                == arm
            )
            & (
                extension_metrics_df[
                    "step"
                ]
                == EXTEND_TO_STEP
            )
        ]

        if len(
            rows
        ):
            selected = (
                rows.iloc[
                    0
                ]
            )

    if selected is None:
        rows = screen_metrics_df[
            (
                screen_metrics_df[
                    "arm"
                ]
                == arm
            )
            & (
                screen_metrics_df[
                    "step"
                ]
                == SCREEN_STEPS
            )
        ]

        if len(
            rows
        ):
            selected = (
                rows.iloc[
                    0
                ]
            )

    if selected is not None:
        final_rows.append(
            selected.to_dict()
        )


final_df = pd.DataFrame(
    final_rows
)

final_df[
    "raw_mask_mean"
] = 0.5 * (
    final_df[
        "raw_d1_mask_dice"
    ]
    + final_df[
        "raw_d0_mask_dice"
    ]
)

final_df[
    "projected_mask_mean"
] = 0.5 * (
    final_df[
        "projected_d1_mask_dice"
    ]
    + final_df[
        "projected_d0_mask_dice"
    ]
)

final_df[
    "raw_mask_gain"
] = (
    final_df[
        "raw_mask_mean"
    ]
    - base_raw
)

final_df[
    "projected_mask_gain"
] = (
    final_df[
        "projected_mask_mean"
    ]
    - base_projected
)

final_df[
    "boundary_auc_gain"
] = (
    final_df[
        "source9_internal_auc"
    ]
    - base_boundary_auc
)

final_df[
    "foreground_ratio"
] = (
    final_df[
        "foreground_loss"
    ]
    / max(
        base_foreground,
        1e-8,
    )
)

final_df[
    "best_instance_gain"
] = np.maximum(
    final_df[
        "raw_mask_gain"
    ],
    final_df[
        "projected_mask_gain"
    ],
)

final_df = final_df.sort_values(
    [
        "best_instance_gain",
        "boundary_auc_gain",
    ],
    ascending=False,
)

display(
    final_df[
        [
            "arm",
            "step",
            "source9_internal_auc",
            "boundary_auc_gain",
            "raw_d1_mask_dice",
            "raw_d0_mask_dice",
            "raw_mask_gain",
            "projected_d1_mask_dice",
            "projected_d0_mask_dice",
            "projected_mask_gain",
            "best_instance_gain",
            "foreground_ratio",
        ]
    ]
)

final_df.to_csv(
    RUN_DIR
    / "final_comparison.csv",
    index=False,
)

# 17. Automatic mechanism interpretation

In [ ]:
findings = []

def get_final(
    arm,
):
    rows = final_df[
        final_df[
            "arm"
        ]
        == arm
    ]

    return (
        rows.iloc[
            0
        ]
        if len(
            rows
        )
        else None
    )


internal = get_final(
    "internal_boundary"
)

raw_d1 = get_final(
    "raw_mask_d1"
)

raw_d0 = get_final(
    "raw_mask_d0"
)

raw_both = get_final(
    "raw_mask_d1d0"
)

projected = get_final(
    "projected_mask_d1d0"
)

combined = get_final(
    "internal_plus_projected_mask"
)


if (
    internal is not None
    and float(
        internal[
            "boundary_auc_gain"
        ]
    )
    >= 0.01
):
    findings.append(
        "Internal-boundary rebalancing reproduces the Notebook-21 positive effect."
    )
else:
    findings.append(
        "Internal-boundary rebalancing did not reproduce its earlier positive movement."
    )


raw_candidates = [
    row
    for row in [
        raw_d1,
        raw_d0,
        raw_both,
    ]
    if row is not None
]

best_raw_gain = (
    max(
        float(
            row[
                "raw_mask_gain"
            ]
        )
        for row in raw_candidates
    )
    if raw_candidates
    else float(
        "-inf"
    )
)

if best_raw_gain >= 0.05:
    findings.append(
        f"Direct binary instance-mask supervision works in the RAW spatial feature space "
        f"(best mean Dice gain={best_raw_gain:+.3f}). "
        "The existing D1/D0 representation can be made instance-aware without requiring "
        "a separate mask projection."
    )
elif best_raw_gain >= 0.02:
    findings.append(
        f"Raw direct-mask supervision shows partial movement "
        f"(best mean Dice gain={best_raw_gain:+.3f}), but not yet a strong mechanism win."
    )
else:
    findings.append(
        f"Raw D1/D0 direct-mask supervision is weak "
        f"(best mean Dice gain={best_raw_gain:+.3f})."
    )


projected_gain = (
    float(
        projected[
            "projected_mask_gain"
        ]
    )
    if projected is not None
    else float(
        "-inf"
    )
)

if projected_gain >= 0.05:
    findings.append(
        f"A dedicated projected mask-feature space works "
        f"(mean projected Dice gain={projected_gain:+.3f})."
    )
elif projected_gain >= 0.02:
    findings.append(
        f"Projected mask supervision shows partial movement "
        f"(gain={projected_gain:+.3f})."
    )
else:
    findings.append(
        f"Projected mask supervision is weak "
        f"(gain={projected_gain:+.3f})."
    )


if (
    projected_gain
    >= 0.05
    and best_raw_gain
    < 0.02
):
    findings.append(
        "The contrast between projected and raw arms indicates that STIR-Net likely needs "
        "an explicitly trained mask-feature projection rather than expecting generic spatial "
        "features to become linearly mask-renderable."
    )


if combined is not None:
    combined_mask_gain = float(
        combined[
            "projected_mask_gain"
        ]
    )

    combined_boundary_gain = float(
        combined[
            "boundary_auc_gain"
        ]
    )

    no_harm = (
        float(
            combined[
                "foreground_ratio"
            ]
        )
        <= 1.10
    )

    if (
        combined_mask_gain >= 0.05
        and combined_boundary_gain >= 0.01
        and no_harm
    ):
        findings.append(
            "Internal-boundary + projected instance-mask supervision is complementary: "
            "both the boundary and mask failures move without >10% foreground-loss regression."
        )


# Verdict.
strong_raw = (
    best_raw_gain
    >= 0.05
)

strong_projected = (
    projected_gain
    >= 0.05
)

strong_combined = (
    combined is not None
    and float(
        combined[
            "projected_mask_gain"
        ]
    )
    >= 0.05
    and float(
        combined[
            "boundary_auc_gain"
        ]
    )
    >= 0.01
    and float(
        combined[
            "foreground_ratio"
        ]
    )
    <= 1.10
)

if strong_combined:
    verdict = (
        "GREEN_COMBINED"
    )
elif strong_raw:
    verdict = (
        "GREEN_RAW_INSTANCE_MASK"
    )
elif strong_projected:
    verdict = (
        "GREEN_PROJECTED_MASK_SPACE"
    )
elif (
    best_raw_gain >= 0.02
    or projected_gain >= 0.02
):
    verdict = (
        "YELLOW"
    )
else:
    verdict = (
        "RED_MASK_OBJECTIVE"
    )


print(
    "=" * 86
)
print(
    "NOTEBOOK 22 — DIRECT INSTANCE-MASK MECHANISM VERDICT"
)
print(
    "=" * 86
)
print(
    "Verdict:",
    verdict
)

for index, statement in enumerate(
    findings,
    start=1,
):
    print(
        f"{index}. {statement}"
    )


report = {
    "verdict": verdict,
    "screen_steps": int(
        SCREEN_STEPS
    ),
    "extended_to_step": int(
        EXTEND_TO_STEP
    ),
    "top_arms_extended": (
        TOP_ARMS
    ),
    "baseline_raw_mask_mean": float(
        base_raw
    ),
    "baseline_projected_mask_mean": float(
        base_projected
    ),
    "best_raw_mask_gain": float(
        best_raw_gain
    ),
    "projected_mask_gain": float(
        projected_gain
    ),
    "findings": findings,
}

with (
    RUN_DIR
    / "mechanism_verdict.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        indent=2,
        default=float,
    )

# 18. Gradient-path sanity table

If a direct-mask scalar decreases but the relevant decoder stage receives negligible gradient, the experiment is not testing what we think it is.

In [ ]:
first_step_gradients = (
    screen_training_df[
        screen_training_df[
            "from_step"
        ]
        == 0
    ][
        [
            "arm",
            "stage_e1_grad_norm",
            "stage_e0_grad_norm",
            "dense_grad_norm",
            "proj_d1_grad_norm",
            "proj_d0_grad_norm",
            "finite_gradients",
            "peak_cuda_gib",
            "step_seconds",
        ]
    ]
)

display(
    first_step_gradients
)

first_step_gradients.to_csv(
    RUN_DIR
    / "first_step_gradient_paths.csv",
    index=False,
)

# 19. Compact plots

In [ ]:
plot_df = final_df.set_index(
    "arm"
)

ax = plot_df[
    [
        "raw_d1_mask_dice",
        "raw_d0_mask_dice",
    ]
].plot(
    kind="bar",
    figsize=(11, 4),
)
ax.set_ylabel(
    "source-9 held-out binary mask soft Dice"
)
ax.set_title(
    "Raw spatial feature instance-mask separability"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.tight_layout()
plt.show()

ax = plot_df[
    [
        "projected_d1_mask_dice",
        "projected_d0_mask_dice",
    ]
].plot(
    kind="bar",
    figsize=(11, 4),
)
ax.set_ylabel(
    "source-9 held-out binary mask soft Dice"
)
ax.set_title(
    "Projected mask-feature instance separability"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.tight_layout()
plt.show()

ax = plot_df[
    "source9_internal_auc"
].plot(
    kind="bar",
    figsize=(10, 4),
)
ax.axhline(
    base_boundary_auc,
    linestyle="--",
)
ax.set_ylabel(
    "source-9 internal-boundary AUC"
)
ax.set_title(
    "Boundary mechanism alongside direct mask supervision"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.tight_layout()
plt.show()

# 20. Save manifest

No full staged overfit should be launched from this notebook.

The next action depends on the mechanism verdict:

```text
GREEN_RAW_INSTANCE_MASK
    → implement direct instance-aware spatial mask supervision

GREEN_PROJECTED_MASK_SPACE
    → implement a dedicated trained mask-feature projection + instance-mask loss

GREEN_COMBINED
    → implement internal-boundary rebalancing + trained mask-feature projection/loss

YELLOW
    → one more bounded weight/formulation test

RED_MASK_OBJECTIVE
    → stop tuning losses and inspect/change E2 → D1/D0 architecture
```

In [ ]:
timing_rows = []

for arm, result in SCREEN_RUNS.items():
    timing_rows.append({
        "phase": "screen",
        "arm": arm,
        "seconds": float(
            result[
                "elapsed_s"
            ]
        ),
    })

for arm, result in EXTENSION_RUNS.items():
    timing_rows.append({
        "phase": "extension",
        "arm": arm,
        "seconds": float(
            result[
                "elapsed_s"
            ]
        ),
    })

timing_df = pd.DataFrame(
    timing_rows
)

timing_df.to_csv(
    RUN_DIR
    / "timing.csv",
    index=False,
)

manifest = {
    "checkpoint": str(
        STEP30_CHECKPOINT
    ),
    "cache_seconds": float(
        cache_seconds
    ),
    "screen_steps": int(
        SCREEN_STEPS
    ),
    "extension_step": int(
        EXTEND_TO_STEP
    ),
    "top_arms_extended": (
        TOP_ARMS
    ),
    "artifacts": [
        path.name
        for path in sorted(
            RUN_DIR.glob(
                "*"
            )
        )
    ],
}

with (
    RUN_DIR
    / "manifest.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

display(
    timing_df
)

print(
    json.dumps(
        manifest,
        indent=2,
    )
)

gc.collect()
torch.cuda.empty_cache()

print(
    "Notebook 22 complete. No full overfit was run."
)

# Decision guide

## `GREEN_RAW_INSTANCE_MASK`

The failure is mainly supervision.

Implement a direct instance-aware binary mask objective on spatial features. The existing spatial representation can be shaped appropriately without a dedicated new mask-feature projection.

## `GREEN_PROJECTED_MASK_SPACE`

Generic D1/D0 features resist becoming mask-renderable, but a small learned projection does.

Implement a dedicated spatial mask-feature projection trained directly with instance-mask supervision. This is also architecturally aligned with STIR-Net's later dot-product mask rendering.

## `GREEN_COMBINED`

The evidence supports two complementary fixes:

```text
1. independently normalized internal cell-cell boundary supervision
2. direct instance-mask supervision in a learned mask-feature space
```

This is the strongest result because Notebook 20 independently identified both failures.

## `YELLOW`

There is causal movement, but not enough to commit a permanent architecture yet. Adjust only the direct-mask loss weight/temperature or extend the winning arm by a few more steps.

## `RED_MASK_OBJECTIVE`

Do not train longer.

If an explicit binary instance-mask objective cannot move held-out source-9 mask Dice in five tail steps, the next target is the E2 → D1/D0 decoder transformation itself rather than loss design.

In [ ]:
# ============================================================================
# FINAL PROBE — FREE LEARNED MASK VECTORS ON FROZEN E2 / D1 / D0 FEATURES
# ============================================================================
#
# Purpose
# -------
# Answer the remaining decisive question:
#
#   "Do the frozen spatial features already contain enough information for an
#    unconstrained learned cell-specific linear mask vector to separate each
#    source-9 GT cell from its touching siblings?"
#
# This is a stronger test than the prototype-cosine experiment above.
#
# For each source-9 GT cell k we learn:
#
#       logit_k(x) = w_k^T F(x) + b_k
#
# where ONLY w_k and b_k are trainable.
#
# The spatial encoder / E2 / D1 / D0 features are completely frozen.
#
# Important:
# A linear mask_feature_proj followed by an arbitrary query mask embedding is
# still equivalent to an arbitrary linear vector in the original feature space.
# Therefore, if this free linear probe cannot separate the cells, merely adding
# another linear 1x1 mask projection cannot create the missing separability.
#
# Evaluation uses held-out voxels from the existing Notebook-22 task split.
#
# Metrics:
#   - held-out AUC                 <- PRIMARY
#   - balanced accuracy
#   - held-out BCE
#   - positive-vs-sibling logit margin
#
# A shuffled-training-label control should remain near AUC ~0.5.
#
# Expected runtime: seconds, not minutes.
# ============================================================================

from dataclasses import dataclass
import gc
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import rankdata


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

FREE_VECTOR_STEPS = 100
FREE_VECTOR_LR = 5e-2
FREE_VECTOR_WEIGHT_DECAY = 1e-4

PROBE_EVAL_STEPS = {
    0,
    1,
    5,
    10,
    25,
    50,
    100,
}

RUN_SHUFFLED_CONTROL = True


# ---------------------------------------------------------------------------
# Sanity: these must already exist from Notebook 22
# ---------------------------------------------------------------------------

_required = [
    "E2_CACHE",
    "SKIP1_CACHE",
    "SKIP0_CACHE",
    "ACQ_CACHE",
    "gt_labels_native",
    "current_labels_native",
    "labels_at_shape",
    "build_instance_mask_tasks",
    "build_fresh_branch",
    "TASKS_D1",
    "TASKS_D0",
    "SOURCE_ID",
    "SEED",
    "device",
    "AMP_DTYPE",
    "RUN_DIR",
]

_missing = [
    name
    for name in _required
    if name not in globals()
]

if _missing:
    raise RuntimeError(
        "Notebook-22 state is missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in _missing
        )
    )

print(
    "Free learned-mask-vector probe preflight OK."
)


# ---------------------------------------------------------------------------
# AUC / balanced-accuracy helpers
# ---------------------------------------------------------------------------

def free_probe_auc(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    positive = (
        labels == 1
    )

    negative = (
        labels == 0
    )

    if (
        positive.sum() == 0
        or negative.sum() == 0
    ):
        return float("nan")

    ranks = rankdata(
        scores
    )

    n_positive = int(
        positive.sum()
    )

    n_negative = int(
        negative.sum()
    )

    return float(
        (
            ranks[
                positive
            ].sum()
            - n_positive
            * (
                n_positive
                + 1
            )
            / 2
        )
        / (
            n_positive
            * n_negative
        )
    )


def balanced_accuracy_from_logits(
    logits,
    labels,
):
    prediction = (
        logits >= 0
    )

    labels = (
        labels > 0.5
    )

    positive = labels
    negative = ~labels

    tpr = (
        (
            prediction[
                positive
            ]
            == labels[
                positive
            ]
        )
        .float()
        .mean()
        if bool(
            positive.any()
        )
        else torch.tensor(
            float("nan"),
            device=logits.device,
        )
    )

    tnr = (
        (
            prediction[
                negative
            ]
            == labels[
                negative
            ]
        )
        .float()
        .mean()
        if bool(
            negative.any()
        )
        else torch.tensor(
            float("nan"),
            device=logits.device,
        )
    )

    return float(
        (
            0.5
            * (
                tpr
                + tnr
            )
        )
        .detach()
        .cpu()
    )


# ---------------------------------------------------------------------------
# Build the same type of source-9 mask tasks at E2
# ---------------------------------------------------------------------------

E2_SHAPE = tuple(
    int(v)
    for v
    in E2_CACHE.shape[
        -3:
    ]
)

GT_E2 = labels_at_shape(
    gt_labels_native,
    E2_SHAPE,
)

CURRENT_E2 = labels_at_shape(
    current_labels_native,
    E2_SHAPE,
)

TASKS_E2 = build_instance_mask_tasks(
    CURRENT_E2,
    GT_E2,
    seed=SEED + 3000,
)

print(
    "Source-9 task counts:"
)

print(
    "  E2:",
    sum(
        task.source_id
        == SOURCE_ID
        for task in TASKS_E2
    ),
)

print(
    "  D1:",
    sum(
        task.source_id
        == SOURCE_ID
        for task in TASKS_D1
    ),
)

print(
    "  D0:",
    sum(
        task.source_id
        == SOURCE_ID
        for task in TASKS_D0
    ),
)


# ---------------------------------------------------------------------------
# Obtain the exact frozen step-30 D1 / D0 features once
# ---------------------------------------------------------------------------

probe_tail = build_fresh_branch()
probe_tail.eval()

torch.cuda.reset_peak_memory_stats()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    FROZEN_D1 = (
        probe_tail.stage_e1(
            E2_CACHE,
            SKIP1_CACHE,
            ACQ_CACHE,
        )
        .detach()
    )

    FROZEN_D0 = (
        probe_tail.stage_e0(
            FROZEN_D1,
            SKIP0_CACHE,
            ACQ_CACHE,
        )
        .detach()
    )

print(
    "Frozen feature shapes:"
)

print(
    "  E2:",
    tuple(
        E2_CACHE.shape
    ),
)

print(
    "  D1:",
    tuple(
        FROZEN_D1.shape
    ),
)

print(
    "  D0:",
    tuple(
        FROZEN_D0.shape
    ),
)

print(
    "Peak CUDA while creating frozen maps:",
    round(
        torch.cuda.max_memory_allocated()
        / 1024**3,
        3,
    ),
    "GiB",
)

del probe_tail
gc.collect()
torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Convert only the source-9 task voxels into compact feature matrices
#
# Once these compact matrices are built, the enormous spatial maps are no
# longer required by the actual linear-probe optimization.
# ---------------------------------------------------------------------------

@dataclass
class FrozenLinearTask:
    gt_id: int

    train_x: torch.Tensor
    train_y: torch.Tensor

    eval_x: torch.Tensor
    eval_y: torch.Tensor


def feature_rows(
    feature,
    indices_cpu,
):
    flat = (
        feature[
            0
        ]
        .flatten(
            1
        )
        .transpose(
            0,
            1,
        )
    )

    indices = indices_cpu.to(
        device=feature.device,
        non_blocking=True,
    )

    return (
        flat[
            indices
        ]
        .float()
        .cpu()
    )


def build_frozen_linear_tasks(
    feature,
    tasks,
):
    result = []

    for task in tasks:
        if int(
            task.source_id
        ) != SOURCE_ID:
            continue

        train_positive = feature_rows(
            feature,
            task.train_pos_indices,
        )

        train_negative = feature_rows(
            feature,
            task.train_neg_indices,
        )

        eval_positive = feature_rows(
            feature,
            task.eval_pos_indices,
        )

        eval_negative = feature_rows(
            feature,
            task.eval_neg_indices,
        )

        train_x = torch.cat(
            [
                train_positive,
                train_negative,
            ],
            dim=0,
        )

        train_y = torch.cat(
            [
                torch.ones(
                    len(
                        train_positive
                    )
                ),
                torch.zeros(
                    len(
                        train_negative
                    )
                ),
            ]
        )

        eval_x = torch.cat(
            [
                eval_positive,
                eval_negative,
            ],
            dim=0,
        )

        eval_y = torch.cat(
            [
                torch.ones(
                    len(
                        eval_positive
                    )
                ),
                torch.zeros(
                    len(
                        eval_negative
                    )
                ),
            ]
        )

        # ---------------------------------------------------------------
        # Per-cell affine standardization.
        #
        # This DOES NOT add nonlinear capacity.
        #
        # A linear classifier after:
        #
        #       x' = (x - mean) / std
        #
        # is still exactly representable as w^T x + b in the original
        # feature coordinates.
        #
        # It only makes the tiny optimization numerically well-conditioned.
        # ---------------------------------------------------------------

        mean = train_x.mean(
            dim=0,
            keepdim=True,
        )

        std = train_x.std(
            dim=0,
            keepdim=True,
        ).clamp_min(
            1e-4
        )

        train_x = (
            train_x
            - mean
        ) / std

        eval_x = (
            eval_x
            - mean
        ) / std

        result.append(
            FrozenLinearTask(
                gt_id=int(
                    task.gt_id
                ),
                train_x=train_x,
                train_y=train_y,
                eval_x=eval_x,
                eval_y=eval_y,
            )
        )

    return result


LINEAR_TASKS = {
    "E2": build_frozen_linear_tasks(
        E2_CACHE,
        TASKS_E2,
    ),
    "D1": build_frozen_linear_tasks(
        FROZEN_D1,
        TASKS_D1,
    ),
    "D0": build_frozen_linear_tasks(
        FROZEN_D0,
        TASKS_D0,
    ),
}

for level, tasks in LINEAR_TASKS.items():
    print(
        level,
        "tasks:",
        len(
            tasks
        ),
        "| channels:",
        (
            tasks[
                0
            ]
            .train_x
            .shape[
                1
            ]
            if tasks
            else 0
        ),
    )

    print(
        "  GT IDs:",
        [
            task.gt_id
            for task in tasks
        ],
    )


# The optimization below only needs the compact task matrices.
del FROZEN_D1
del FROZEN_D0

gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA after compact extraction:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3,
    ),
    "GiB",
)


# ---------------------------------------------------------------------------
# Train completely free mask vectors
# ---------------------------------------------------------------------------

def evaluate_free_vectors(
    weight,
    bias,
    tasks,
    *,
    level,
    step,
    control,
):
    per_cell = []

    for task_index, task in enumerate(
        tasks
    ):
        train_x = task.train_x.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        train_y = task.train_y.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        eval_x = task.eval_x.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        eval_y = task.eval_y.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        train_logits = (
            train_x
            @ weight[
                task_index
            ]
            + bias[
                task_index
            ]
        )

        eval_logits = (
            eval_x
            @ weight[
                task_index
            ]
            + bias[
                task_index
            ]
        )

        train_auc = free_probe_auc(
            train_logits
            .detach()
            .cpu()
            .numpy(),
            train_y
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.int64
            ),
        )

        eval_auc = free_probe_auc(
            eval_logits
            .detach()
            .cpu()
            .numpy(),
            eval_y
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.int64
            ),
        )

        eval_bce = float(
            F.binary_cross_entropy_with_logits(
                eval_logits,
                eval_y,
            )
            .detach()
            .cpu()
        )

        balanced_accuracy = (
            balanced_accuracy_from_logits(
                eval_logits,
                eval_y,
            )
        )

        positive_logits = (
            eval_logits[
                eval_y > 0.5
            ]
        )

        negative_logits = (
            eval_logits[
                eval_y < 0.5
            ]
        )

        margin = float(
            (
                positive_logits.mean()
                - negative_logits.mean()
            )
            .detach()
            .cpu()
        )

        positive_probability = float(
            positive_logits.sigmoid()
            .mean()
            .detach()
            .cpu()
        )

        negative_probability = float(
            negative_logits.sigmoid()
            .mean()
            .detach()
            .cpu()
        )

        per_cell.append({
            "control": control,
            "level": level,
            "step": int(
                step
            ),
            "gt_id": int(
                task.gt_id
            ),
            "train_auc": float(
                train_auc
            ),
            "eval_auc": float(
                eval_auc
            ),
            "eval_balanced_accuracy": float(
                balanced_accuracy
            ),
            "eval_bce": float(
                eval_bce
            ),
            "eval_logit_margin": float(
                margin
            ),
            "eval_positive_probability": float(
                positive_probability
            ),
            "eval_sibling_probability": float(
                negative_probability
            ),
        })

    frame = pd.DataFrame(
        per_cell
    )

    summary = {
        "control": control,
        "level": level,
        "step": int(
            step
        ),
        "cell_count": int(
            len(
                frame
            )
        ),
        "mean_train_auc": float(
            frame[
                "train_auc"
            ].mean()
        ),
        "mean_eval_auc": float(
            frame[
                "eval_auc"
            ].mean()
        ),
        "median_eval_auc": float(
            frame[
                "eval_auc"
            ].median()
        ),
        "min_eval_auc": float(
            frame[
                "eval_auc"
            ].min()
        ),
        "mean_balanced_accuracy": float(
            frame[
                "eval_balanced_accuracy"
            ].mean()
        ),
        "mean_eval_bce": float(
            frame[
                "eval_bce"
            ].mean()
        ),
        "mean_logit_margin": float(
            frame[
                "eval_logit_margin"
            ].mean()
        ),
        "mean_positive_probability": float(
            frame[
                "eval_positive_probability"
            ].mean()
        ),
        "mean_sibling_probability": float(
            frame[
                "eval_sibling_probability"
            ].mean()
        ),
    }

    return (
        summary,
        frame,
    )


def train_free_mask_vectors(
    level,
    tasks,
    *,
    shuffle_training_labels=False,
):
    if not tasks:
        raise RuntimeError(
            f"No tasks available for {level}."
        )

    channel_count = int(
        tasks[
            0
        ].train_x.shape[
            1
        ]
    )

    task_count = len(
        tasks
    )

    # ---------------------------------------------------------------
    # Only these parameters are learned.
    # ---------------------------------------------------------------

    weight = torch.nn.Parameter(
        torch.zeros(
            (
                task_count,
                channel_count,
            ),
            device=device,
            dtype=torch.float32,
        )
    )

    bias = torch.nn.Parameter(
        torch.zeros(
            (
                task_count,
            ),
            device=device,
            dtype=torch.float32,
        )
    )

    optimizer = torch.optim.AdamW(
        [
            weight,
            bias,
        ],
        lr=float(
            FREE_VECTOR_LR
        ),
        weight_decay=float(
            FREE_VECTOR_WEIGHT_DECAY
        ),
    )

    # ---------------------------------------------------------------
    # Move only tiny compact matrices to the GPU.
    # ---------------------------------------------------------------

    train_x = [
        task.train_x.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )
        for task in tasks
    ]

    true_train_y = [
        task.train_y.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )
        for task in tasks
    ]

    if shuffle_training_labels:
        train_y = []

        generator = torch.Generator(
            device="cpu"
        )

        generator.manual_seed(
            SEED
            + 7000
            + sum(
                ord(
                    character
                )
                for character in level
            )
        )

        for labels in true_train_y:
            permutation = torch.randperm(
                len(
                    labels
                ),
                generator=generator,
            ).to(
                device
            )

            train_y.append(
                labels[
                    permutation
                ]
            )
    else:
        train_y = true_train_y

    control = (
        "shuffled_train_labels"
        if shuffle_training_labels
        else "true_labels"
    )

    summary_rows = []
    per_cell_frames = []
    training_rows = []

    start = time.perf_counter()

    for step in range(
        FREE_VECTOR_STEPS
        + 1
    ):
        if step in PROBE_EVAL_STEPS:
            with torch.no_grad():
                summary, per_cell = (
                    evaluate_free_vectors(
                        weight,
                        bias,
                        tasks,
                        level=level,
                        step=step,
                        control=control,
                    )
                )

            summary_rows.append(
                summary
            )

            per_cell_frames.append(
                per_cell
            )

            print(
                f"{control:22s} | "
                f"{level:2s} | "
                f"step {step:3d} | "
                f"train AUC {summary['mean_train_auc']:.4f} | "
                f"eval AUC {summary['mean_eval_auc']:.4f} | "
                f"bal acc {summary['mean_balanced_accuracy']:.4f} | "
                f"margin {summary['mean_logit_margin']:+.4f}"
            )

        if step == FREE_VECTOR_STEPS:
            break

        optimizer.zero_grad(
            set_to_none=True
        )

        losses = []

        for task_index in range(
            task_count
        ):
            logits = (
                train_x[
                    task_index
                ]
                @ weight[
                    task_index
                ]
                + bias[
                    task_index
                ]
            )

            labels = train_y[
                task_index
            ]

            losses.append(
                F.binary_cross_entropy_with_logits(
                    logits,
                    labels,
                )
            )

        loss = torch.stack(
            losses
        ).mean()

        loss.backward()

        finite = (
            bool(
                torch.isfinite(
                    weight.grad
                ).all()
            )
            and bool(
                torch.isfinite(
                    bias.grad
                ).all()
            )
        )

        grad_norm = float(
            torch.sqrt(
                weight.grad
                .square()
                .sum()
                + bias.grad
                .square()
                .sum()
            )
            .detach()
            .cpu()
        )

        optimizer.step()

        training_rows.append({
            "control": control,
            "level": level,
            "from_step": int(
                step
            ),
            "to_step": int(
                step + 1
            ),
            "loss": float(
                loss.detach().cpu()
            ),
            "grad_norm": float(
                grad_norm
            ),
            "finite_gradients": bool(
                finite
            ),
        })

    elapsed = (
        time.perf_counter()
        - start
    )

    del (
        weight,
        bias,
        optimizer,
        train_x,
        true_train_y,
        train_y,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "summary": pd.DataFrame(
            summary_rows
        ),
        "per_cell": pd.concat(
            per_cell_frames,
            ignore_index=True,
        ),
        "training": pd.DataFrame(
            training_rows
        ),
        "elapsed_s": float(
            elapsed
        ),
    }


# ---------------------------------------------------------------------------
# Run E2 / D1 / D0
# ---------------------------------------------------------------------------

FREE_VECTOR_RUNS = {}

summary_frames = []
per_cell_frames = []
training_frames = []
timing_rows = []

for level in [
    "E2",
    "D1",
    "D0",
]:
    print(
        "\n"
        + "=" * 88
    )
    print(
        "FREE MASK VECTOR PROBE:",
        level
    )
    print(
        "=" * 88
    )

    result = train_free_mask_vectors(
        level,
        LINEAR_TASKS[
            level
        ],
        shuffle_training_labels=False,
    )

    FREE_VECTOR_RUNS[
        (
            level,
            "true_labels",
        )
    ] = result

    summary_frames.append(
        result[
            "summary"
        ]
    )

    per_cell_frames.append(
        result[
            "per_cell"
        ]
    )

    training_frames.append(
        result[
            "training"
        ]
    )

    timing_rows.append({
        "level": level,
        "control": "true_labels",
        "seconds": float(
            result[
                "elapsed_s"
            ]
        ),
    })

    if RUN_SHUFFLED_CONTROL:
        print(
            "\n"
            + "-" * 88
        )
        print(
            "SHUFFLED-LABEL CONTROL:",
            level
        )
        print(
            "-" * 88
        )

        shuffled = (
            train_free_mask_vectors(
                level,
                LINEAR_TASKS[
                    level
                ],
                shuffle_training_labels=True,
            )
        )

        FREE_VECTOR_RUNS[
            (
                level,
                "shuffled_train_labels",
            )
        ] = shuffled

        summary_frames.append(
            shuffled[
                "summary"
            ]
        )

        per_cell_frames.append(
            shuffled[
                "per_cell"
            ]
        )

        training_frames.append(
            shuffled[
                "training"
            ]
        )

        timing_rows.append({
            "level": level,
            "control": "shuffled_train_labels",
            "seconds": float(
                shuffled[
                    "elapsed_s"
                ]
            ),
        })


free_vector_summary_df = pd.concat(
    summary_frames,
    ignore_index=True,
)

free_vector_per_cell_df = pd.concat(
    per_cell_frames,
    ignore_index=True,
)

free_vector_training_df = pd.concat(
    training_frames,
    ignore_index=True,
)

free_vector_timing_df = pd.DataFrame(
    timing_rows
)


# ---------------------------------------------------------------------------
# Final true-label results
# ---------------------------------------------------------------------------

final_true = (
    free_vector_summary_df[
        (
            free_vector_summary_df[
                "control"
            ]
            == "true_labels"
        )
        & (
            free_vector_summary_df[
                "step"
            ]
            == FREE_VECTOR_STEPS
        )
    ]
    .copy()
    .sort_values(
        "level"
    )
)

print(
    "\n"
    + "=" * 88
)
print(
    "FINAL FREE MASK VECTOR RESULTS"
)
print(
    "=" * 88
)

display(
    final_true[
        [
            "level",
            "cell_count",
            "mean_train_auc",
            "mean_eval_auc",
            "median_eval_auc",
            "min_eval_auc",
            "mean_balanced_accuracy",
            "mean_eval_bce",
            "mean_logit_margin",
            "mean_positive_probability",
            "mean_sibling_probability",
        ]
    ]
)


# ---------------------------------------------------------------------------
# Per-cell final AUC table
# ---------------------------------------------------------------------------

final_per_cell = (
    free_vector_per_cell_df[
        (
            free_vector_per_cell_df[
                "control"
            ]
            == "true_labels"
        )
        & (
            free_vector_per_cell_df[
                "step"
            ]
            == FREE_VECTOR_STEPS
        )
    ]
    .copy()
)

print(
    "\nPer-cell held-out AUC:"
)

display(
    final_per_cell.pivot(
        index="gt_id",
        columns="level",
        values="eval_auc",
    )
)


# ---------------------------------------------------------------------------
# Shuffled-control result
# ---------------------------------------------------------------------------

if RUN_SHUFFLED_CONTROL:
    final_shuffled = (
        free_vector_summary_df[
            (
                free_vector_summary_df[
                    "control"
                ]
                == "shuffled_train_labels"
            )
            & (
                free_vector_summary_df[
                    "step"
                ]
                == FREE_VECTOR_STEPS
            )
        ]
        .copy()
    )

    print(
        "\nShuffled-label control:"
    )

    display(
        final_shuffled[
            [
                "level",
                "mean_train_auc",
                "mean_eval_auc",
                "mean_balanced_accuracy",
                "mean_logit_margin",
            ]
        ]
    )


# ---------------------------------------------------------------------------
# Automatic interpretation
# ---------------------------------------------------------------------------

level_auc = {
    str(
        row[
            "level"
        ]
    ): float(
        row[
            "mean_eval_auc"
        ]
    )
    for _, row
    in final_true.iterrows()
}

e2_auc = level_auc.get(
    "E2",
    float(
        "nan"
    ),
)

d1_auc = level_auc.get(
    "D1",
    float(
        "nan"
    ),
)

d0_auc = level_auc.get(
    "D0",
    float(
        "nan"
    ),
)


if (
    e2_auc >= 0.80
    and d1_auc < e2_auc - 0.10
):
    verdict = (
        "E2_GOOD_DECODER_LOSSES_INSTANCE_INFORMATION"
    )

    interpretation = (
        "E2 already contains strong linearly renderable cell identity, "
        "but E2→D1 loses a substantial fraction of it. "
        "The next fix should target preservation of E2 instance information "
        "through stage_e1 / spatial decoder training."
    )

elif (
    d1_auc >= 0.80
    and d0_auc >= 0.80
):
    verdict = (
        "SPATIAL_FEATURES_ARE_MASK_RENDERABLE"
    )

    interpretation = (
        "Frozen D1/D0 features can already support strong independent cell-specific "
        "linear masks. The main remaining failure is therefore mask-query/readout "
        "training rather than spatial representational capacity."
    )

elif (
    e2_auc < 0.70
    and d1_auc < 0.70
    and d0_auc < 0.70
):
    verdict = (
        "SPATIAL_REPRESENTATION_NOT_INSTANCE_SEPARABLE"
    )

    interpretation = (
        "Even completely free learned mask vectors cannot reliably separate the "
        "touching source-9 cells. The spatial representation itself needs stronger "
        "instance-aware supervision and/or an upstream architectural change."
    )

elif (
    e2_auc >= 0.75
    and d1_auc < 0.70
):
    verdict = (
        "PARTIAL_E2_SIGNAL_DECODER_DEGRADATION"
    )

    interpretation = (
        "E2 contains useful but not overwhelming instance-separating information, "
        "and the high-resolution decoder weakens it further. "
        "Prioritize E2→D1 preservation together with explicit instance-aware supervision."
    )

elif (
    d1_auc >= 0.75
    or d0_auc >= 0.75
):
    verdict = (
        "FEATURES_PARTIALLY_MASK_RENDERABLE"
    )

    interpretation = (
        "The frozen spatial features contain substantial linear instance information. "
        "The mask-query/readout path is likely underusing that information, although "
        "the representation is not yet ideal."
    )

else:
    verdict = (
        "MIXED_RESULT"
    )

    interpretation = (
        "Linear instance separability is neither clearly strong nor completely absent. "
        "Use the E2/D1/D0 AUC trajectory and per-cell table to choose whether decoder "
        "preservation or stronger upstream instance supervision is the dominant next fix."
    )


print(
    "\n"
    + "=" * 88
)
print(
    "LEARNED MASK VECTOR PROBE VERDICT"
)
print(
    "=" * 88
)
print(
    "Verdict:",
    verdict
)
print()
print(
    interpretation
)

print()
print(
    f"E2 held-out AUC: {e2_auc:.4f}"
)
print(
    f"D1 held-out AUC: {d1_auc:.4f}"
)
print(
    f"D0 held-out AUC: {d0_auc:.4f}"
)


# ---------------------------------------------------------------------------
# Save artifacts
# ---------------------------------------------------------------------------

free_vector_summary_df.to_csv(
    RUN_DIR
    / "free_mask_vector_summary.csv",
    index=False,
)

free_vector_per_cell_df.to_csv(
    RUN_DIR
    / "free_mask_vector_per_cell.csv",
    index=False,
)

free_vector_training_df.to_csv(
    RUN_DIR
    / "free_mask_vector_training.csv",
    index=False,
)

free_vector_timing_df.to_csv(
    RUN_DIR
    / "free_mask_vector_timing.csv",
    index=False,
)

probe_report = {
    "verdict": verdict,
    "interpretation": interpretation,
    "steps": int(
        FREE_VECTOR_STEPS
    ),
    "lr": float(
        FREE_VECTOR_LR
    ),
    "E2_mean_eval_auc": float(
        e2_auc
    ),
    "D1_mean_eval_auc": float(
        d1_auc
    ),
    "D0_mean_eval_auc": float(
        d0_auc
    ),
    "shuffled_control": bool(
        RUN_SHUFFLED_CONTROL
    ),
}

with (
    RUN_DIR
    / "free_mask_vector_verdict.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        probe_report,
        handle,
        indent=2,
    )


print(
    "\nSaved:"
)
print(
    " ",
    RUN_DIR
    / "free_mask_vector_summary.csv",
)
print(
    " ",
    RUN_DIR
    / "free_mask_vector_per_cell.csv",
)
print(
    " ",
    RUN_DIR
    / "free_mask_vector_training.csv",
)
print(
    " ",
    RUN_DIR
    / "free_mask_vector_timing.csv",
)
print(
    " ",
    RUN_DIR
    / "free_mask_vector_verdict.json",
)

display(
    free_vector_timing_df
)

gc.collect()
torch.cuda.empty_cache()

print(
    "\nFinal learned-mask-vector probe complete."
)